# Offline d1/d2/d3 validation harness

Scores any retrieval method on synthetic proxies of all three difficulty levels,
built from dataset1's labelled pairs. **Run All** and read the table at the bottom.

- Lives in `amine/` alongside `eval_harness.py` (kernel cwd = this folder).
- Cell 2 symlinks `data/` and `out/` here so the data + outputs show in the left panel.
- Edit `EMBEDDERS` in `eval_harness.py` to plug in new methods, then re-run.
- Last cell gives download links for the outputs.

In [ ]:
import subprocess, sys
# nibabel+scipy for the reference embedders; torch+monai for the learned/Swin embedder.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel>=5.3', 'scipy', 'monai>=1.3'], check=True)
print('deps ready')

In [ ]:
import os
# Make the input data and output dir browsable in the file panel on the left:
# create symlinks INSIDE this folder (amine/) pointing at the real locations.
# One-time per container; harmless if they already exist.
for name, target in [('data', '/workspace/data/ehl'), ('out', '/workspace/out')]:
    if os.path.islink(name) or os.path.exists(name):
        print('exists:', name, '->', os.path.realpath(name))
        continue
    if os.path.isdir(target):
        os.symlink(target, name)
        print('linked:', name, '->', target)
    else:
        print('SKIP', name, '- target not found:', target, '(fix the path if your mount differs)')

In [ ]:
import os
os.environ['DATA_ROOT'] = '/workspace/data/ehl'
os.environ.setdefault('N_VAL', '60')   # held-out dataset1 pairs to score on
os.environ.setdefault('GRID', '96')    # resample cube size

# --- LEARNED MODEL: MIND-input contrastive embedder (the d2/d3 model) ---------
# Feeds modality-invariant MIND fields to a 3D CNN trained CLIP-style with heavy
# rigid+elastic+resection augmentation -> a deformation-robust global embedding.
os.environ['SKIP_LEARNED'] = '1'         # skip the old raw-intensity Swin/CNN embedder
os.environ['USE_MIND_LEARNED'] = '1'     # train + score mind_embedder (the new model)
# knobs (defaults are sensible; bump epochs if loss is still falling):
# os.environ['MIND_EPOCHS']='200'; os.environ['MIND_RESECT']='0.5'; os.environ['MIND_TTA']='8'

# also score the training-free GPU rankers for reference (nmi/mind/dense_mind):
assert os.path.isdir(os.environ['DATA_ROOT']), 'data root not found - fix DATA_ROOT above'
for f in ('eval_harness.py','rankers.py','mind_embedder.py','learned_embedder.py'):
    assert os.path.exists(f), f're-upload {f} here'
print('ready: training mind_learned + scoring rankers on', os.environ['DATA_ROOT'])

In [ ]:
import importlib, eval_harness
importlib.reload(eval_harness)   # pick up edits without restarting the kernel
results = eval_harness.evaluate()

In [ ]:
import os, shutil
from IPython.display import FileLink, display
# Click these links to download outputs to your computer.
# (submission.csv is produced by run_baseline.ipynb; searched in a few common spots.)
def offer(fname, candidates):
    for c in candidates:
        if os.path.exists(c):
            if os.path.abspath(c) != os.path.abspath(fname):
                shutil.copy(c, fname)   # bring it next to this notebook so the link resolves
            display(FileLink(fname))
            return
    print('not found yet:', fname)

offer('eval_results.md', ['eval_results.md'])
offer('submission.csv', ['submission.csv', 'out/submission.csv',
                         '/workspace/out/submission.csv', '/root/submission.csv'])

In [ ]:
from IPython.display import Markdown
Markdown(open('eval_results.md').read())